# 01. Исследование датасета КТГ

Работаем пошагово. Сейчас реализована **только загрузка исходного файла**. Следующие разделы — задания для совместного выполнения, без готового анализа и заранее принятых решений.

Исходный ZIP скачивается отдельно командой `just fetch_dataset`. Здесь его не изменяем, не очищаем строки, не делим данные и не обучаем модели.

## 1. Загрузка исходного файла

**Что делаем:** открываем `CTG.xls` непосредственно из `runtime/data/raw/cardiotocography.zip`, затем читаем лист `Data` в pandas.

**Зачем:** получить исходную таблицу для исследования без дополнительных копий и преобразований.

В листе `Data` первая строка содержит номера признаков, вторая — названия колонок, поэтому используем `header=1`. Повторяющиеся названия pandas автоматически различит суффиксами вроде `.1`; это ещё предстоит разобрать. Пустые и служебные колонки пока сохраняем.

**Ожидаемый результат:** переменная `dataset` с исходной таблицей и первые пять строк для проверки чтения. Наличие строки в этой таблице ещё не означает, что это корректная запись для обучения.

In [1]:
from io import BytesIO
from zipfile import ZipFile

import pandas as pd

from classifier.data.settings import DataSettings

In [2]:
settings = DataSettings()
archive_path = settings.raw_dir / "cardiotocography.zip"

with ZipFile(archive_path) as archive:
    workbook = BytesIO(archive.read("CTG.xls"))

excel_file = pd.ExcelFile(workbook, engine="xlrd")
dataset = pd.read_excel(excel_file, sheet_name="Data", header=1)

dataset.head()

,b,e,AC,FM,UC,DL,DS,DP,DR,Unnamed: 9,...,E,AD,DE,LD,FS,SUSP,Unnamed: 42,CLASS,Unnamed: 44,NSP
0,240.0,357.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,...,-1.0,-1.0,-1.0,-1.0,1.0,-1.0,NaN,9.0,NaN,2.0
1,5.0,632.0,4.0,0.0,4.0,2.0,0.0,0.0,0.0,NaN,...,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,NaN,6.0,NaN,1.0
2,177.0,779.0,2.0,0.0,5.0,2.0,0.0,0.0,0.0,NaN,...,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,NaN,6.0,NaN,1.0
3,411.0,1192.0,2.0,0.0,6.0,2.0,0.0,0.0,0.0,NaN,...,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,NaN,6.0,NaN,1.0
4,533.0,1147.0,4.0,0.0,5.0,0.0,0.0,0.0,0.0,NaN,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,NaN,2.0,NaN,1.0


## 2. Происхождение данных и смысл одной записи

**Что сделать:** Прочитать лист `Description`, посмотреть назначение `Raw Data` и сверить документацию первоисточника. Выяснить, как получены измерения и экспертные метки, что означает строка и могут ли несколько строк относиться к одному исследованию.

**Зачем:** Понять, что именно будет классифицировать сервис и какие ограничения есть у данных.

**Ожидаемый результат:** Короткое описание источника, единицы наблюдения и происхождения CLASS/NSP. Не приравнивать ID файла к ID пациента без подтверждения.

## 3. Структура таблицы и словарь признаков

**Что сделать:** Посмотреть размер, типы и имена колонок. Разобрать повторяющиеся заголовки, пустые разделители и служебные строки. Составить таблицу: колонка, смысл, единицы, роль. По документации выделить 21 входной признак, две цели и служебные поля.

**Зачем:** Не включить в модель лишние колонки или скрытые представления целевых меток.

**Ожидаемый результат:** Явный список входов и целей с объяснением выбора. Пока не удалять столбцы автоматически.

## 4. Пропуски и корректность значений

**Что сделать:** Посчитать количество и долю пропусков, проверить числовые типы, значения CLASS/NSP и полностью пустые строки. Отдельно разобраться в нулях: ноль не обязательно означает пропуск.

**Зачем:** Отделить ошибки чтения и служебные строки от реальных особенностей измерений.

**Ожидаемый результат:** Таблица проблем с примерами исходных строк и предложениями обработки. Пропуски пока не заполняем.

## 5. Дубликаты, конфликтующие метки и группы

**Что сделать:** Проверить дубли всей строки, совпадения 21 входного признака и совпадения признаков вместе с метками. Найти одинаковые входы с разными CLASS/NSP. Проверить доступность групп исследований и число записей в группе.

**Зачем:** Не завысить качество из-за попадания связанных или одинаковых записей в разные выборки.

**Ожидаемый результат:** Количество и примеры повторов/конфликтов, возможный ключ группы. Удаление записей — только после обсуждения.

## 6. Обсуждение правил подготовки и разбиения

**Что сделать:** Совместно выбрать обработку проблемных строк и принцип train/validation/test. Рассмотреть ориентир 60/20/20, фиксированный seed, баланс классов и целостность групп.

**Зачем:** Сначала понять структуру, затем отделить независимую оценку от исследовательских решений.

**Ожидаемый результат:** Согласованные правила. Только после этого реализовать разбиение и проверить отсутствие пересечений; не подбирать seed ради хороших метрик.

## 7. Баланс CLASS и NSP на train

**Что сделать:** Для каждой цели построить таблицу количества и долей, столбчатый график; отметить редкие классы.

**Зачем:** Понять ограничения обучения и почему одной accuracy может быть недостаточно.

**Ожидаемый результат:** Вывод о дисбалансе и предложение метрик: macro-F1 и precision/recall по классам. Пока не балансировать данные.

## 8. Связь CLASS и NSP на train

**Что сделать:** Разобрать определения классов. Построить таблицу CLASS × NSP: количества и доли внутри CLASS. Проверить, соответствует ли каждому паттерну ровно один NSP, и изучить исключения.

**Зачем:** Проверить гипотезу «предсказываем паттерн, затем получаем NSP», а не принимать её заранее.

**Ожидаемый результат:** Обоснованное предложение: NSP напрямую, CLASS с отображением в NSP или два выхода модели. Соответствие в выборке не доказывает универсальное медицинское правило; выведенное правило позднее проверяем на validation.

## 9. Распределения и необычные значения на train

**Что сделать:** Составить сводку числовых признаков, построить гистограммы и boxplot. Проверить масштабы, асимметрию, постоянные признаки, долю нулей и экстремальные значения.

**Зачем:** Подготовить понятные гипотезы о масштабировании и качестве входных данных.

**Ожидаемый результат:** Выводы по распределениям. Значения за границами IQR не удаляем автоматически: они могут быть важными наблюдениями.

## 10. Взаимосвязи признаков на train

**Что сделать:** Построить корреляции входных признаков, разобрать сильно связанные пары; сравнить распределения нескольких понятных признаков по целевым категориям.

**Зачем:** Найти избыточность и возможный сигнал для классификации.

**Ожидаемый результат:** Обоснованные гипотезы для будущих экспериментов. Коды CLASS/NSP не трактуем как непрерывные числа, корреляцию не считаем причинностью.

## 11. Итоги EDA и следующий шаг

**Что сделать:** Собрать факты и решения: входы, цели, проблемы данных, split, необходимые преобразования и метрики. Отдельно записать оставшиеся вопросы.

**Зачем:** Перейти к обучению с понятной задачей и правилами оценки.

**Ожидаемый результат:** Краткие выводы в notebook и `experiments/README.md`. После обсуждения — подготовка воспроизводимого ML-пайплайна и первые модели. Scaler обучаем только на train; test не используем для выбора решений.